# U.S. Airline Flight Delay Analysis

**Team Members:** Jenny Chen, Sanjana Kosuru, Tianqi Zhen  
**Course:** CS-GY 6513-C Big Data  
**Semester:** Spring 2026

This notebook presents the Data Engineering pipeline for a large-scale U.S. airline delay analysis project using PySpark.

---



## Project Overview

The goal of this project is to analyze large-scale U.S. airline on-time performance data and identify important delay patterns across airlines, airports, and time periods.  
The project also investigates major causes of delays and prepares a clean feature-rich dataset for downstream SQL analytics and machine learning.

## Data Engineering Overview

The Data Engineering stage is designed to transform raw airline performance files into a clean and structured dataset suitable for large-scale analysis.

This stage includes:

- extracting nested compressed files
- identifying valid monthly CSV files
- loading data with PySpark
- selecting core attributes
- removing invalid or incomplete records
- handling missing values
- creating derived features for analysis and prediction

The result is a cleaned dataset that supports both Spark SQL analysis and machine learning tasks.

## Environment Setup

The project uses PySpark as the main big data processing framework.  
PySpark allows us to efficiently load, clean, and process millions of flight records in a distributed environment.

In [1]:
!pip install pyspark -q

## Raw Data Extraction

The raw dataset is stored as a compressed outer ZIP archive.  
After extracting the outer archive, we obtain a folder containing monthly ZIP files for the selected year.  
These monthly ZIP files are then extracted into a separate directory for further processing.

In [3]:
import os
import zipfile
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lpad, floor

outer_zip_path = "/content/BTS_Data_2025.zip"
outer_extract_dir = "/content/bts_data"
os.makedirs(outer_extract_dir, exist_ok=True)

# Extract outer zip
with zipfile.ZipFile(outer_zip_path, 'r') as zip_ref:
    zip_ref.extractall(outer_extract_dir)

print("Outer zip extracted.")
print("Contents of outer_extract_dir:")
for item in os.listdir(outer_extract_dir):
    print(item)

Outer zip extracted.
Contents of outer_extract_dir:
BTS_Data_2025
__MACOSX


In [4]:
for root, dirs, files in os.walk(outer_extract_dir):
    print("Folder:", root)
    print("Files:", files)
    print("-----")

Folder: /content/bts_data
Files: []
-----
Folder: /content/bts_data/BTS_Data_2025
Files: ['On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip', 'On_Time_Reporting_Carrier_On_T

In [5]:
# Folder that contains the monthly zip files
monthly_zip_dir = "/content/bts_data/BTS_Data_2025"

# Folder to extract all monthly zips
monthly_extract_dir = "/content/bts_data/extracted_monthly"
os.makedirs(monthly_extract_dir, exist_ok=True)

# Collect all inner zip files, and ignore duplicate files like "... 2.zip"
monthly_zip_files = [
    f for f in os.listdir(monthly_zip_dir)
    if f.endswith(".zip") and " 2" not in f
]

print("Monthly zip files to process:", len(monthly_zip_files))
for f in monthly_zip_files:
    print(f)

# Extract each monthly zip into its own subfolder
for zip_file in monthly_zip_files:
    zip_path = os.path.join(monthly_zip_dir, zip_file)

    # Use zip filename (without extension) as subfolder name
    folder_name = os.path.splitext(zip_file)[0].replace(" ", "_")
    target_dir = os.path.join(monthly_extract_dir, folder_name)
    os.makedirs(target_dir, exist_ok=True)

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f"Extracted: {zip_file}")
    except Exception as e:
        print(f"Failed to extract {zip_file}: {e}")

Monthly zip files to process: 12
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip
Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip
Extracted: On_Time_Reporting_Carrier_On_Ti

## File Discovery and Validation

After extracting the monthly archives, we recursively scan the extracted directory and identify valid CSV files only.  
System-generated files such as `__MACOSX` artifacts are excluded to avoid loading invalid data into Spark.

In [6]:
csv_files = []

for root, dirs, files in os.walk(monthly_extract_dir):
    for file in files:
        full_path = os.path.join(root, file)
        if file.endswith(".csv") and "__MACOSX" not in full_path and "/._" not in full_path:
            csv_files.append(full_path)

csv_files = sorted(csv_files)

print("Valid CSV files found:", len(csv_files))
print("First few CSV files:")
for f in csv_files[:5]:
    print(f)

Valid CSV files found: 12
First few CSV files:
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_10.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_11.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12/On_Time_Reporting_Carrier_On_Time_Performance_(1987_prese

## Spark Data Ingestion

The validated CSV files are loaded into a Spark DataFrame using schema inference and header recognition.  
This allows us to work with the airline dataset in a scalable way while preserving typed columns such as dates, integers, and delay measures.

In [7]:
spark = SparkSession.builder \
    .appName("BTS_Airline_Delay_Cleaning") \
    .getOrCreate()

print("Spark session is ready.")

Spark session is ready.


In [8]:
df = spark.read.csv(
    csv_files,
    header=True,
    inferSchema=True
)

# Preview data
df.show(5, truncate=False)

# Print schema
df.printSchema()

# Check dataset size
print("Row count:", df.count())
print("Column count:", len(df.columns))
print("All columns:")
for c in df.columns:
    print(c)

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+--------------+----------------+----

## Core Column Selection

The raw airline dataset contains many fields.  
To keep the analysis focused and efficient, we select a subset of core columns related to:

- flight date and schedule
- airline identity
- origin and destination airports
- departure and arrival delays
- cancellation and diversion indicators
- elapsed time and distance
- detailed delay causes

In [9]:
core_columns = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek",
    "FlightDate",

    "Reporting_Airline", "DOT_ID_Reporting_Airline", "IATA_CODE_Reporting_Airline",

    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",

    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes",

    "Cancelled", "CancellationCode", "Diverted",

    "CRSElapsedTime", "ActualElapsedTime", "AirTime", "Distance",

    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]

df_core = df.select(*core_columns)

print("Selected columns:", len(df_core.columns))
df_core.show(5, truncate=False)

Selected columns: 35
+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+------+--------------+-----------+----+------------+---------+----------+-------+--------+---------------+----------+-------+--------+---------------+---------+----------------+--------+--------------+-----------------+-------+--------+------------+------------+--------+-------------+-----------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|DOT_ID_Reporting_Airline|IATA_CODE_Reporting_Airline|Origin|OriginCityName|OriginState|Dest|DestCityName|DestState|CRSDepTime|DepTime|DepDelay|DepDelayMinutes|CRSArrTime|ArrTime|ArrDelay|ArrDelayMinutes|Cancelled|CancellationCode|Diverted|CRSElapsedTime|ActualElapsedTime|AirTime|Distance|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+--

## Data Cleaning

The data cleaning process removes incomplete and invalid records and standardizes important delay-related fields.

The cleaning steps include:

- removing rows with critical missing values
- excluding cancelled flights from standard delay analysis
- filtering out invalid distance and departure hour values
- replacing missing delay minutes with zero
- standardizing delay cause columns

In [10]:
from pyspark.sql.functions import col, when, lpad, floor

df_clean = df_core \
    .withColumn("is_cancelled", col("Cancelled").cast("int")) \
    .withColumn("is_diverted", col("Diverted").cast("int")) \
    .withColumn("is_dep_delayed", when(col("DepDelayMinutes") > 15, 1).otherwise(0)) \
    .withColumn("is_arr_delayed", when(col("ArrDelayMinutes") > 15, 1).otherwise(0)) \
    .withColumn("CRSDepTime_PADDED", lpad(col("CRSDepTime").cast("string"), 4, "0")) \
    .withColumn("dep_hour", floor(col("CRSDepTime_PADDED").substr(1, 2).cast("int"))) \
    .withColumn(
        "season",
        when(col("Month").isin(12, 1, 2), "Winter")
        .when(col("Month").isin(3, 4, 5), "Spring")
        .when(col("Month").isin(6, 7, 8), "Summer")
        .otherwise("Fall")
    )

df_clean = df_clean.dropna(subset=[
    "FlightDate", "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "CRSArrTime", "Distance"
])

df_clean = df_clean.filter(col("is_cancelled") == 0)
df_clean = df_clean.filter(col("Distance") > 0)
df_clean = df_clean.filter((col("dep_hour") >= 0) & (col("dep_hour") <= 23))

df_clean = df_clean.withColumn(
    "DepDelayMinutes",
    when(col("DepDelayMinutes").isNull(), 0).otherwise(col("DepDelayMinutes"))
)

df_clean = df_clean.withColumn(
    "ArrDelayMinutes",
    when(col("ArrDelayMinutes").isNull(), 0).otherwise(col("ArrDelayMinutes"))
)

delay_cols = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
for c in delay_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c).isNull(), 0).otherwise(col(c))
    )

df_clean = df_clean.withColumn(
    "is_weekend",
    when(col("DayOfWeek").isin(6, 7), 1).otherwise(0)
)

df_clean = df_clean.withColumn(
    "dep_period",
    when((col("dep_hour") >= 5) & (col("dep_hour") < 12), "Morning")
    .when((col("dep_hour") >= 12) & (col("dep_hour") < 17), "Afternoon")
    .when((col("dep_hour") >= 17) & (col("dep_hour") < 21), "Evening")
    .otherwise("Night")
)

df_clean = df_clean.withColumn(
    "is_long_flight",
    when(col("Distance") >= 1500, 1).otherwise(0)
)

In [11]:
df_clean.createOrReplaceTempView("airline_data")
print("SQL table 'airline_data' created.")

SQL table 'airline_data' created.


## Cleaned Data Validation

After cleaning and feature engineering, we validate the resulting dataset by checking:

- final row count and column count
- sample records
- null counts for important columns

This confirms that the cleaned dataset is ready for downstream SQL analytics and machine learning.

In [12]:
print("Cleaned row count:", df_clean.count())
print("Cleaned column count:", len(df_clean.columns))

Cleaned row count: 6969618
Cleaned column count: 45


In [13]:
df_clean.select(
    "Year", "Month", "DayOfWeek",
    "Reporting_Airline",
    "Origin", "Dest",
    "CRSDepTime", "dep_hour",
    "DepDelayMinutes", "ArrDelayMinutes",
    "is_dep_delayed", "is_arr_delayed",
    "is_cancelled", "is_diverted",
    "season"
).show(10, False)

+----+-----+---------+-----------------+------+----+----------+--------+---------------+---------------+--------------+--------------+------------+-----------+------+
|Year|Month|DayOfWeek|Reporting_Airline|Origin|Dest|CRSDepTime|dep_hour|DepDelayMinutes|ArrDelayMinutes|is_dep_delayed|is_arr_delayed|is_cancelled|is_diverted|season|
+----+-----+---------+-----------------+------+----+----------+--------+---------------+---------------+--------------+--------------+------------+-----------+------+
|2024|1    |1        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter|
|2024|1    |2        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter|
|2024|1    |3        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter

In [14]:
for c in ["Origin", "Dest", "Distance", "DepDelayMinutes", "ArrDelayMinutes"]:
    print(c, df_clean.filter(col(c).isNull()).count())

Origin 0
Dest 0
Distance 0
DepDelayMinutes 0
ArrDelayMinutes 0


## Data Engineering Summary

At the end of the Data Engineering stage, we have transformed raw compressed airline performance files into a cleaned and structured Spark dataset.

Key outcomes of this stage include:

- successful extraction of nested monthly archives
- validation of 12 monthly CSV files
- scalable ingestion with PySpark
- selection of 35 core variables
- cleaning of missing and invalid records
- creation of derived features for analytics and modeling
- final cleaned dataset with millions of records ready for analysis

This engineered dataset provides the foundation for the next stage of the project: large-scale SQL analysis of airline delay patterns.

## Problem Objectives

In this project, we aim to better understand flight delays using a large-scale airline dataset by exploring the following questions:

- How often do flight delays happen overall?
- Which airlines tend to perform worse in terms of delays?
- Which airports experience the most delays, and why?
- How do delays change depending on time, such as during the day, across the week, or by season?
- What are the main reasons behind flight delays?
- Do delays build up throughout the day?
- How do different factors interact with each other, such as airline and time of day?
- Do long-distance flights behave differently from shorter ones?
- Are delays mainly caused by airline operations or external factors like weather?
- Can we use this data to build a simple model to predict whether a flight will be delayed?

## 1. Overall Flight Delay Pattern

In this section, we first look at the overall delay situation in the dataset.

The goal is to understand how frequently delays occur and what the average delay looks like across all flights.

In [15]:
spark.sql("""
SELECT
    COUNT(*) AS total_flights,
    AVG(DepDelayMinutes) AS avg_dep_delay,
    AVG(ArrDelayMinutes) AS avg_arr_delay,
    SUM(is_dep_delayed) / COUNT(*) AS dep_delay_rate,
    SUM(is_arr_delayed) / COUNT(*) AS arr_delay_rate
FROM airline_data
""").show()

+-------------+-----------------+-----------------+-------------------+-------------------+
|total_flights|    avg_dep_delay|    avg_arr_delay|     dep_delay_rate|     arr_delay_rate|
+-------------+-----------------+-----------------+-------------------+-------------------+
|      6969618|16.32227964287282|16.27152305908301|0.20401663333628903|0.20552087646697423|
+-------------+-----------------+-----------------+-------------------+-------------------+



To further understand delay distribution, we also categorize flights based on delay severity.

In [16]:
spark.sql("""
SELECT
    CASE
        WHEN ArrDelayMinutes <= 0 THEN 'On Time / Early'
        WHEN ArrDelayMinutes <= 15 THEN 'Minor Delay'
        WHEN ArrDelayMinutes <= 60 THEN 'Moderate Delay'
        ELSE 'Severe Delay'
    END AS delay_category,
    COUNT(*) AS flight_count
FROM airline_data
GROUP BY delay_category
ORDER BY flight_count DESC
""").show()

+---------------+------------+
| delay_category|flight_count|
+---------------+------------+
|On Time / Early|     4416687|
|    Minor Delay|     1120529|
| Moderate Delay|      915470|
|   Severe Delay|      516932|
+---------------+------------+



## 2. Airline Performance Comparison

In this section, we compare the performance of different airlines.

The goal is to identify which airlines tend to have higher delay rates and worse average delays.


In [17]:
spark.sql("""
SELECT
    Reporting_Airline,
    COUNT(*) AS total_flights,
    AVG(ArrDelayMinutes) AS avg_arr_delay,
    SUM(is_arr_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY Reporting_Airline
ORDER BY delay_rate DESC
LIMIT 10
""").show()

+-----------------+-------------+------------------+-------------------+
|Reporting_Airline|total_flights|     avg_arr_delay|         delay_rate|
+-----------------+-------------+------------------+-------------------+
|               F9|       202866|24.582507665158282| 0.2857797758126054|
|               AA|       971564|23.494062151335374|0.25626206817049624|
|               B6|       235992|21.306823960134242|0.25226278856910406|
|               NK|       248901| 18.22827951675566|0.23751210320569222|
|               OH|       223343|20.111577260088744|0.21747715397393247|
|               AS|       241900|12.692224059528732|0.21628772219925588|
|               G4|       115686| 19.90275400653493| 0.2144771190982487|
|               MQ|       278679|14.579993469188565|0.20417397794595216|
|               WN|      1409366|12.383495841392513|0.20195321868130778|
|               UA|       749879|15.589153716799643| 0.1952301637997597|
+-----------------+-------------+------------------

To ensure the results are reliable, we also filter out airlines with a very small number of flights.

In [18]:
spark.sql("""
SELECT
    Reporting_Airline,
    COUNT(*) AS total_flights,
    AVG(ArrDelayMinutes) AS avg_arr_delay,
    SUM(is_arr_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY Reporting_Airline
HAVING COUNT(*) >= 10000
ORDER BY delay_rate DESC
LIMIT 10
""").show()

+-----------------+-------------+------------------+-------------------+
|Reporting_Airline|total_flights|     avg_arr_delay|         delay_rate|
+-----------------+-------------+------------------+-------------------+
|               F9|       202866|24.582507665158282| 0.2857797758126054|
|               AA|       971564|23.494062151335374|0.25626206817049624|
|               B6|       235992|21.306823960134242|0.25226278856910406|
|               NK|       248901| 18.22827951675566|0.23751210320569222|
|               OH|       223343|20.111577260088744|0.21747715397393247|
|               AS|       241900|12.692224059528732|0.21628772219925588|
|               G4|       115686| 19.90275400653493| 0.2144771190982487|
|               MQ|       278679|14.579993469188565|0.20417397794595216|
|               WN|      1409366|12.383495841392513|0.20195321868130778|
|               UA|       749879|15.589153716799643| 0.1952301637997597|
+-----------------+-------------+------------------

## 3. Airport Congestion Analysis

In this section, we examine airport-level delay performance.

The goal is to identify which airports experience the most delays and whether this may be related to traffic volume or congestion.

In [19]:
spark.sql("""
SELECT
    Origin,
    COUNT(*) AS total_flights,
    AVG(DepDelayMinutes) AS avg_dep_delay,
    SUM(is_dep_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY Origin
ORDER BY delay_rate DESC
LIMIT 10
""").show()

+------+-------------+------------------+-------------------+
|Origin|total_flights|     avg_dep_delay|         delay_rate|
+------+-------------+------------------+-------------------+
|   VRB|           41|48.292682926829265| 0.4878048780487805|
|   HTS|          422|  44.6303317535545| 0.4312796208530806|
|   EWN|           58| 68.08620689655173|0.41379310344827586|
|   HGR|          288|          36.34375| 0.3715277777777778|
|   ALO|           51|26.627450980392158|0.35294117647058826|
|   CKB|          203|40.206896551724135|0.33004926108374383|
|   OTH|          340| 37.94705882352941|0.32941176470588235|
|   USA|          692|33.664739884393065|0.32514450867052025|
|   EAU|           50|             44.74|               0.32|
|   GUF|           41| 62.63414634146341| 0.3170731707317073|
+------+-------------+------------------+-------------------+



To make the comparison more reliable, we also focus on airports with a larger number of flights.

In [20]:
spark.sql("""
SELECT
    Origin,
    COUNT(*) AS total_flights,
    AVG(DepDelayMinutes) AS avg_dep_delay,
    SUM(is_dep_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY Origin
HAVING COUNT(*) >= 10000
ORDER BY delay_rate DESC
LIMIT 10
""").show()

+------+-------------+------------------+-------------------+
|Origin|total_flights|     avg_dep_delay|         delay_rate|
+------+-------------+------------------+-------------------+
|   BWI|        97324|17.268289425013357| 0.2778862356664338|
|   FLL|        90918|20.756571855958118| 0.2728502606744539|
|   DFW|       308078|  21.2843403293971|0.27057758100221374|
|   MIA|       108158| 21.93286673200318|0.26322602119122024|
|   CLT|       211657|21.003028484765444|0.25926380889835915|
|   DAL|        71753|14.937702953186626|0.24396192493693644|
|   MCO|       157163|19.110973956974608|0.24395054815700898|
|   MDW|        78495|15.137843174724505|0.23996432893814892|
|   SJU|        34821|21.757818557766864| 0.2366388099135579|
|   IAH|       114038|20.208491906206703|  0.234220172223294|
+------+-------------+------------------+-------------------+



We also look at the busiest airports in the dataset to better understand whether high traffic volume is associated with delay performance.

In [21]:
spark.sql("""
SELECT
    Origin,
    COUNT(*) AS total_flights,
    AVG(DepDelayMinutes) AS avg_dep_delay,
    SUM(is_dep_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY Origin
ORDER BY total_flights DESC
LIMIT 10
""").show()

+------+-------------+------------------+-------------------+
|Origin|total_flights|     avg_dep_delay|         delay_rate|
+------+-------------+------------------+-------------------+
|   ATL|       334706| 14.14917569449009| 0.1942510740769511|
|   DFW|       308078|  21.2843403293971|0.27057758100221374|
|   DEN|       305201|16.409667727169964|0.23153266208171008|
|   ORD|       280821| 19.45722364068214| 0.2279459157256758|
|   CLT|       211657|21.003028484765444|0.25926380889835915|
|   PHX|       192685|13.950836858084438|0.19817318421257493|
|   LAX|       191946|13.961598574599106|0.17910245589905494|
|   LAS|       186526|15.889870581045002| 0.2314154595069856|
|   SEA|       162268|12.850364828555229|0.20243670964084107|
|   MCO|       157163|19.110973956974608|0.24395054815700898|
+------+-------------+------------------+-------------------+



## 4. Temporal Delay Patterns

In this section, we analyze how flight delays change over time.

The goal is to understand how delays vary during the day, across the week, and by season.

We first examine how delays vary by departure hour.

In [22]:
spark.sql("""
SELECT
    dep_hour,
    AVG(DepDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY dep_hour
ORDER BY dep_hour
""").show()

+--------+------------------+
|dep_hour|         avg_delay|
+--------+------------------+
|       0|18.707025780413474|
|       1| 19.59441423628275|
|       2|15.618543046357615|
|       3|16.248785228377066|
|       4| 20.06553911205074|
|       5| 9.460136338200133|
|       6| 8.480940773193868|
|       7| 9.187175822850286|
|       8| 9.814214408948835|
|       9|11.043015697817022|
|      10|12.444978433295125|
|      11| 13.64457571319029|
|      12|15.030844155844155|
|      13|16.367149145284817|
|      14| 17.99460018640726|
|      15|19.146688007328176|
|      16|20.533590560589467|
|      17|21.519169581560917|
|      18|23.144932573661688|
|      19|24.151616690632295|
+--------+------------------+
only showing top 20 rows


Next, we look at how delays vary across different days of the week.

In [23]:
spark.sql("""
SELECT
    DayOfWeek,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY DayOfWeek
ORDER BY DayOfWeek
""").show()

+---------+------------------+
|DayOfWeek|         avg_delay|
+---------+------------------+
|        1|16.602810172799035|
|        2|14.454087080562507|
|        3|14.245572500401918|
|        4|16.345005835435817|
|        5|18.073351307812146|
|        6|15.548388567007422|
|        7|18.354684578836437|
+---------+------------------+



We also analyze how delays differ across seasons.

In [24]:
spark.sql("""
SELECT
    season,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY season
ORDER BY avg_delay DESC
""").show()

+------+------------------+
|season|         avg_delay|
+------+------------------+
|Summer|20.902780029444497|
|Winter|17.289430736375312|
|Spring|16.991898295584825|
|  Fall| 9.813366108098808|
+------+------------------+



Finally, we group flights into different periods of the day to better understand delay patterns.

In [25]:
spark.sql("""
SELECT
    dep_period,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY dep_period
ORDER BY avg_delay DESC
""").show()

+----------+------------------+
|dep_period|         avg_delay|
+----------+------------------+
|   Evening|22.781374861444075|
|     Night|21.027412739511007|
| Afternoon|17.712589193847798|
|   Morning|11.026403188822774|
+----------+------------------+



## 5. Delay Cause Analysis

In this section, we analyze the main causes of flight delays.

The goal is to understand which factors contribute the most to delays.

We first look at the average contribution of each delay category across all flights.

In [26]:
spark.sql("""
SELECT
    AVG(CarrierDelay) AS carrier_delay,
    AVG(WeatherDelay) AS weather_delay,
    AVG(NASDelay) AS nas_delay,
    AVG(SecurityDelay) AS security_delay,
    AVG(LateAircraftDelay) AS late_aircraft_delay
FROM airline_data
""").show()

+-----------------+------------------+-----------------+--------------------+-------------------+
|    carrier_delay|     weather_delay|        nas_delay|      security_delay|late_aircraft_delay|
+-----------------+------------------+-----------------+--------------------+-------------------+
|5.254456843976241|0.8991926099823548|2.904462769695556|0.026393125132539545|  6.189231174506264|
+-----------------+------------------+-----------------+--------------------+-------------------+



Next, we compare delay causes across different airlines.

In [27]:
spark.sql("""
SELECT
    Reporting_Airline,
    AVG(CarrierDelay) AS carrier_delay,
    AVG(WeatherDelay) AS weather_delay,
    AVG(LateAircraftDelay) AS late_aircraft_delay
FROM airline_data
GROUP BY Reporting_Airline
ORDER BY carrier_delay DESC
LIMIT 10
""").show()

+-----------------+------------------+-------------------+-------------------+
|Reporting_Airline|     carrier_delay|      weather_delay|late_aircraft_delay|
+-----------------+------------------+-------------------+-------------------+
|               OO| 8.059640630272543|  2.416435493554031| 3.1922664632072237|
|               B6| 8.011364792026848|  0.508440964100478|  7.751127156852775|
|               AA|  7.40125611899988| 1.1731507136946202| 10.923771362462999|
|               G4| 6.489229465968224| 1.9519129367425618|  7.084055114707052|
|               F9|6.2866522729289285| 0.4636656709354944| 12.786213559689648|
|               DL| 6.079497083359594| 0.5900907254736601|  3.814959649222846|
|               OH|5.4571667793483565|  1.548564315872895|  9.618250851828801|
|               NK| 4.631692922085488|0.45351766364940277|  4.887947416844448|
|               HA|4.5835113639288565| 0.2173045738848421| 2.8268770430619545|
|               UA|4.2422844218867315| 0.76708375617

We also examine how delay causes vary across different periods of the day.

In [28]:
spark.sql("""
SELECT
    dep_period,
    AVG(CarrierDelay) AS carrier_delay,
    AVG(WeatherDelay) AS weather_delay,
    AVG(LateAircraftDelay) AS late_aircraft_delay
FROM airline_data
GROUP BY dep_period
ORDER BY dep_period
""").show()

+----------+-----------------+------------------+-------------------+
|dep_period|    carrier_delay|     weather_delay|late_aircraft_delay|
+----------+-----------------+------------------+-------------------+
| Afternoon|5.017448630927872|0.9441374545861926|  7.216193308287154|
|   Evening|6.169788363880081| 1.263720903255742| 10.413189879617457|
|   Morning|4.537422890250113| 0.625590117314309|  2.803706900015488|
|     Night|7.697009152448052|  1.19569119143251|  8.767658659779144|
+----------+-----------------+------------------+-------------------+



Finally, we group delay causes into operational and external factors for comparison.

In [29]:
spark.sql("""
SELECT
    AVG(CarrierDelay + LateAircraftDelay) AS operational_delay,
    AVG(WeatherDelay + NASDelay + SecurityDelay) AS external_delay
FROM airline_data
""").show()

+------------------+----------------+
| operational_delay|  external_delay|
+------------------+----------------+
|11.443688018482504|3.83004850481045|
+------------------+----------------+



## 6. Delay Accumulation Effect

In this section, we examine whether delays tend to build up throughout the day.

The goal is to understand if flights later in the day experience higher delays compared to earlier flights.

We first analyze how average delay changes across different departure hours.

In [30]:
spark.sql("""
SELECT
    dep_hour,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY dep_hour
ORDER BY dep_hour
""").show()

+--------+------------------+
|dep_hour|         avg_delay|
+--------+------------------+
|       0|16.332592043489004|
|       1|18.401384083044984|
|       2|13.435761589403974|
|       3|14.767735665694849|
|       4|19.382663847780126|
|       5| 9.837823676044158|
|       6| 9.214318663137673|
|       7|10.134937428826696|
|       8|10.484823582929964|
|       9|11.402787997710735|
|      10|12.403085626953548|
|      11|13.474792115304263|
|      12|14.811705221861471|
|      13|16.191675088489006|
|      14|17.950079835281947|
|      15| 19.11122450757585|
|      16|20.647423578611967|
|      17| 21.78306543794425|
|      18|22.866173631628143|
|      19|23.357101961663787|
+--------+------------------+
only showing top 20 rows


We also group flights into different periods of the day to better observe delay accumulation.

In [31]:
spark.sql("""
SELECT
    dep_period,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY dep_period
ORDER BY avg_delay DESC
""").show()

+----------+------------------+
|dep_period|         avg_delay|
+----------+------------------+
|   Evening|22.781374861444075|
|     Night|21.027412739511007|
| Afternoon|17.712589193847798|
|   Morning|11.026403188822774|
+----------+------------------+



Finally, we compare delay rates across different periods of the day.

In [32]:
spark.sql("""
SELECT
    dep_period,
    SUM(is_arr_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY dep_period
ORDER BY delay_rate DESC
""").show()

+----------+-------------------+
|dep_period|         delay_rate|
+----------+-------------------+
|   Evening|0.29018315159951535|
|     Night| 0.2640594898503196|
| Afternoon|0.22861435926363052|
|   Morning|0.13480070372788272|
+----------+-------------------+



## 7. Multi-Dimensional Delay Analysis

In this section, we analyze how different factors interact with each other.

The goal is to understand how combinations of variables such as airline, time of day, and season influence flight delays.

We first examine how delay patterns vary across airlines and different periods of the day.

In [33]:
spark.sql("""
SELECT
    Reporting_Airline,
    dep_period,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY Reporting_Airline, dep_period
ORDER BY avg_delay DESC
""").show()

+-----------------+----------+------------------+
|Reporting_Airline|dep_period|         avg_delay|
+-----------------+----------+------------------+
|               G4|     Night| 40.79146009915241|
|               F9|   Evening|33.140279753437646|
|               AA|   Evening| 32.92000952767931|
|               F9|     Night| 32.02711978214646|
|               G4|   Evening|30.514267379679143|
|               B6|   Evening| 29.64693554068334|
|               B6|     Night|28.630288565849426|
|               OH|   Evening|28.155509231685528|
|               AA|     Night| 27.42267445677148|
|               AA| Afternoon|26.114985249193285|
|               F9| Afternoon|  25.8879824840202|
|               OH|     Night|25.775515251442705|
|               NK|   Evening|  25.6332162746976|
|               B6| Afternoon| 24.98679331618726|
|               NK|     Night|22.427678308402342|
|               UA|   Evening|22.305017692441556|
|               WN|     Night|21.984308945598173|


Next, we analyze how delay patterns differ across seasons and periods of the day.

In [34]:
spark.sql("""
SELECT
    season,
    dep_period,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY season, dep_period
ORDER BY avg_delay DESC
""").show()

+------+----------+------------------+
|season|dep_period|         avg_delay|
+------+----------+------------------+
|Summer|   Evening|31.943538043451554|
|Summer|     Night|29.534384802035863|
|Spring|   Evening| 23.78941852009729|
|Summer| Afternoon|22.503435389972758|
|Winter|   Evening|21.770171218019865|
|Spring|     Night|21.031077205824104|
|Winter|     Night|19.661064647825913|
|Spring| Afternoon| 19.09883551531625|
|Winter| Afternoon| 18.68602014668382|
|Winter|   Morning|13.606492269600533|
|  Fall|   Evening|13.133165533506249|
|Summer|   Morning|12.369548912179674|
|  Fall|     Night|11.693501848318077|
|Spring|   Morning|  11.1027942115402|
|  Fall| Afternoon|10.639553993091928|
|  Fall|   Morning| 7.208088143063886|
+------+----------+------------------+



We also explore how delay patterns vary across both airports and airlines.

In [35]:
spark.sql("""
SELECT
    Origin,
    Reporting_Airline,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY Origin, Reporting_Airline
ORDER BY avg_delay DESC
LIMIT 20
""").show()

+------+-----------------+------------------+
|Origin|Reporting_Airline|         avg_delay|
+------+-----------------+------------------+
|   HDN|               B6|111.08474576271186|
|   MGW|               OO|107.11764705882354|
|   MVY|               MQ|105.22727272727273|
|   RSW|               G4| 96.45833333333333|
|   ORD|               OH| 91.84242424242424|
|   RDM|               AA| 90.51538461538462|
|   MHK|               OO| 87.91428571428571|
|   SAT|               G4| 78.31818181818181|
|   CRP|               OH|             74.45|
|   RNO|               G4|              73.0|
|   AVP|               OO|  71.6046511627907|
|   TVC|               AA| 65.89010989010988|
|   LAS|               MQ|  65.8076923076923|
|   EWN|               OH|  62.3448275862069|
|   GUF|               G4| 59.02439024390244|
|   LFT|               NK|              59.0|
|   HYA|               B6| 58.91752577319588|
|   STS|               AA| 58.91208791208791|
|   COU|               OO| 56.6113

To focus on more meaningful results, we also restrict the analysis to airports with a large number of flights.

In [36]:
spark.sql("""
SELECT
    Origin,
    Reporting_Airline,
    COUNT(*) AS total_flights,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY Origin, Reporting_Airline
HAVING COUNT(*) >= 5000
ORDER BY avg_delay DESC
LIMIT 20
""").show()

+------+-----------------+-------------+------------------+
|Origin|Reporting_Airline|total_flights|         avg_delay|
+------+-----------------+-------------+------------------+
|   ASE|               OO|         7446|29.988315874294923|
|   ATL|               F9|        11939|29.276405059050173|
|   SJU|               F9|         5620| 28.18238434163701|
|   IAH|               AA|         6952| 27.96964902186421|
|   BNA|               AA|         8379| 27.76500775748896|
|   AUS|               AA|        13668|27.452151009657594|
|   FLL|               AA|         6510| 27.14592933947773|
|   PHL|               F9|        11437|26.746524438226807|
|   DFW|               F9|         9996|26.591036414565828|
|   MSY|               AA|         6700| 25.98044776119403|
|   SFO|               AA|        10598|25.818833742215514|
|   ATL|               AA|         7553|25.777969018932875|
|   PBI|               AA|         5967|25.716608010725658|
|   DFW|               AA|       163528|

## 8. Flight Distance Impact

In this section, we examine how flight distance affects delay patterns.

The goal is to understand whether long-distance flights behave differently from short-distance flights in terms of delays.

We first compare average delays between long-distance and short-distance flights.

In [37]:
spark.sql("""
SELECT
    is_long_flight,
    COUNT(*) AS total_flights,
    AVG(ArrDelayMinutes) AS avg_delay,
    SUM(is_arr_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY is_long_flight
""").show()

+--------------+-------------+------------------+-------------------+
|is_long_flight|total_flights|         avg_delay|         delay_rate|
+--------------+-------------+------------------+-------------------+
|             1|       905521|16.060088059802037| 0.2099321826881983|
|             0|      6064097|16.303095580430195|0.20486215837246666|
+--------------+-------------+------------------+-------------------+



We also examine how flight distance interacts with time of day.

In [38]:
spark.sql("""
SELECT
    is_long_flight,
    dep_period,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY is_long_flight, dep_period
ORDER BY avg_delay DESC
""").show()

+--------------+----------+------------------+
|is_long_flight|dep_period|         avg_delay|
+--------------+----------+------------------+
|             0|   Evening| 22.84628998640099|
|             0|     Night|22.489209479586975|
|             1|   Evening|22.248595213936902|
|             1| Afternoon| 18.27782017944781|
|             0| Afternoon|17.642206721567277|
|             1|     Night| 16.86416725923383|
|             1|   Morning|11.844344602769707|
|             0|   Morning| 10.90018187310268|
+--------------+----------+------------------+



Finally, we analyze delay patterns across different distance ranges.

In [39]:
spark.sql("""
SELECT
    CASE
        WHEN Distance < 500 THEN 'Short (<500)'
        WHEN Distance < 1500 THEN 'Medium (500-1500)'
        ELSE 'Long (>1500)'
    END AS distance_range,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY distance_range
ORDER BY avg_delay DESC
""").show()

+-----------------+------------------+
|   distance_range|         avg_delay|
+-----------------+------------------+
|Medium (500-1500)|16.844223536301385|
|     Long (>1500)|16.060088059802037|
|     Short (<500)| 15.46196338968886|
+-----------------+------------------+



## 9. Operational vs External Delay Analysis

In this section, we compare operational and external causes of flight delays.

The goal is to understand whether delays are mainly driven by airline operations or by external factors such as weather.

We first compare the overall contribution of operational and external delay factors.

In [40]:
spark.sql("""
SELECT
    AVG(CarrierDelay + LateAircraftDelay) AS operational_delay,
    AVG(WeatherDelay + NASDelay + SecurityDelay) AS external_delay
FROM airline_data
""").show()

+------------------+----------------+
| operational_delay|  external_delay|
+------------------+----------------+
|11.443688018482504|3.83004850481045|
+------------------+----------------+



Next, we examine how operational and external delays vary across different airlines.

In [41]:
spark.sql("""
SELECT
    Reporting_Airline,
    AVG(CarrierDelay + LateAircraftDelay) AS operational_delay,
    AVG(WeatherDelay + NASDelay + SecurityDelay) AS external_delay
FROM airline_data
GROUP BY Reporting_Airline
ORDER BY operational_delay DESC
LIMIT 10
""").show()

+-----------------+------------------+-----------------+
|Reporting_Airline| operational_delay|   external_delay|
+-----------------+------------------+-----------------+
|               F9|19.072865832618575|4.504687823489397|
|               AA| 18.32502748146288|4.132313465710957|
|               B6|15.762491948879623|4.642449744059121|
|               OH|15.075417631177158|4.055157314086405|
|               G4|13.573284580675276|5.463997372197154|
|               OO|11.251907093479767| 4.94670437713358|
|               UA|10.300775191730933|4.362770526978353|
|               DL| 9.894456732582439|3.030046748278577|
|               NK| 9.519640338929936|7.796047424478005|
|               MQ| 9.223970948654186|4.318574417160963|
+-----------------+------------------+-----------------+



We also analyze how operational and external delays change across different periods of the day.

In [42]:
spark.sql("""
SELECT
    dep_period,
    AVG(CarrierDelay + LateAircraftDelay) AS operational_delay,
    AVG(WeatherDelay + NASDelay + SecurityDelay) AS external_delay
FROM airline_data
GROUP BY dep_period
ORDER BY dep_period
""").show()

+----------+------------------+------------------+
|dep_period| operational_delay|    external_delay|
+----------+------------------+------------------+
| Afternoon|12.233641939215026| 4.413502236172436|
|   Evening| 16.58297824349754| 5.098591885133916|
|   Morning| 7.341129790265601| 2.796825532011342|
|     Night|16.464667812227194|3.5174117412166024|
+----------+------------------+------------------+



Finally, we compare the relative proportion of operational and external delays.

In [43]:
spark.sql("""
SELECT
    AVG(CarrierDelay + LateAircraftDelay) /
    (AVG(CarrierDelay + LateAircraftDelay) + AVG(WeatherDelay + NASDelay + SecurityDelay))
    AS operational_ratio
FROM airline_data
""").show()

+------------------+
| operational_ratio|
+------------------+
|0.7492395852861873|
+------------------+

